In [ ]:
# Proximal Pruning 
This notebook is used to prune spurious branches that result from proximal branches. 
![proximal_pruning.png](attachment:eab0c016-a1fd-4787-bbe1-56bed34c51b1.png)

There are 3 main steps. First, we define a "proximal region" in the lung based on the widths of the branches (eg. above 100um in widths that is the largest connected component). We select for the largest connected component because there are some thick tips that are not continuous with the proximal region. 
Then, we create an image stacks with just end tips. We compare the image of proximal region, and we delete end tips within this region. 

In [ ]:
from skimage.morphology import skeletonize 
from skimage import data 
from skimage.morphology import medial_axis, skeletonize, skeletonize_3d
import tifffile as tf
import time 
import numpy as np 
import skan 
import napari
import os
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd
from scipy.ndimage import label
import sys

In [ ]:
# Import images
folder_path = r"F:\Megumi\DBOX-EQS1 Dropbox\Eqs1 Box03\Megumi-Cadisha\Skeleton by Lobe\Right Upper-Middle Lobes\proximal_pruning\EH3453RUM-P6.3(6.9)\\"
skeleton_img = tf.imread(folder_path + "skeleton.tif")
thickness_img = tf.imread(folder_path + "Tb_corrected.tif") 
voxel = int(input("voxel:"))
prox_width = int(input("width:")) 

# If PCW7.0-7.9, use 120 
# If PCW8.0-10.9, use 100
# If PCW11.0-11.9, use 80 (EH4413)
# If late PCW11 -- GW13a, use 70
# for EH3822LU, EH3879-2 lumen, GW13 lumen use 40
# for EH3685LU lumen, use 60
# for PCW19 use40

In [ ]:
####### PART 1: Define a Proximal Region #######
# Here we apply a threshold 
thresholded_stack = thickness_img >= prox_width
# Apply connected components
labeled_stack, num_features = label(thresholded_stack)
# Find the sizes of connected components
sizes = []
for i in range(1, num_features + 1):
    sizes.append(np.sum(labeled_stack == i))

######### IF you want to apply just the LARGEST ######### 
# then apply connnected components and keep the largest OR two largest (sometimes we need this) 

# Find index of largest component

# Find the index of the largest component
largest_index = np.argsort(sizes)[-1]

# Create an empty array to hold the largest component
largest_component = np.zeros_like(labeled_stack, dtype=np.uint8)

# Set pixels of the largest component to 255
largest_component[labeled_stack == largest_index + 1] = 255  # Add 1 because label values start from 1

# Save the largest component
tf.imwrite(os.path.join(folder_path, f"proximal_img_{prox_width}.tif"), largest_component) 
'''
######### Alternatively keep the TWO largest ######### 

# Find the indices of the two largest components
largest_indices = np.argsort(sizes)[-2:]

# Create an empty array to hold the merged components
merged_components = np.zeros_like(labeled_stack, dtype=np.uint8)

# Merge the two largest components
for label_value in largest_indices:
    merged_components[labeled_stack == label_value + 1] = 255  # Add 1 because label values start from 1

# Save the merged components
tf.imwrite(os.path.join(folder_path, f"proximal_img_{prox_width}.tif"), merged_components) 
'''
print("finished")
# then proceed to the next step 

In [ ]:
### Here we check in napari (in 3D) that the proximal_img is sufficient (sometimes there is a break in a main branch, and we can see that the proximal_img does not reach some parts of the data) 
# Also check that they are one connected piece. If you need to take the 2 largest, you can rerun the previous cell. 
import napari
viewer = napari.Viewer(ndisplay=3)
prox_img = tf.imread(folder_path + f"proximal_img_{prox_width}.tif")
viewer.add_image(prox_img, name = ('proximal_img'), colormap = 'bop orange')
viewer.add_image(thickness_img, name=('thickness_img'), opacity = 0.4)

In [ ]:
####### PART 2 and 3: Delete end tips within the proximal region #######
# We need to skeletenize the skeleton to be able to read it as a skeleton object using skan.
proximal_img = tf.imread(folder_path + f"proximal_img_{prox_width}.tif")

# Make a copy of the skeleton image
skeleton_copy = skeleton_img.copy()

# Save the copy as "proximal_pruned.tif" -- this will be our starting skeleton 
# while branch number is decreasing (because we delete proximal branches), we update this skeleton. 
tf.imwrite(folder_path +  f"proximal_pruned_{prox_width}.tif", skeleton_copy)

# We found that applying the code multiple times is necessary since only applying it once leaves some branches. 
# Here, we count the number of iterations, and continuously prune within the proximal region while total branch number changes. 
iteration = 0 

# We count the total number of branches with this variable. We first set it to None, but this gets updated later. 
prev_num_rows = None

# While total branch number changes between iterations: 
while True:
    iteration += 1
    print("iteration:", iteration)
    skeleton_img = tf.imread(folder_path +  f"proximal_pruned_{prox_width}.tif") 
    
    # Skeletonize the input image
    binary_skeleton = skeletonize_3d(skeleton_img)
    skeleton = skan.Skeleton(binary_skeleton, spacing=voxel)
    endpoint_skel = np.copy(binary_skeleton)
       
    # Create a paths table for the skeleton
    paths_table = skan.summarize(skeleton)
    print("total branches:", len(paths_table))
    paths_table['path-id'] = np.arange(skeleton.n_paths)
    paths_table['random-path-id'] = np.random.default_rng().permutation(skeleton.n_paths)
    
    # Check if the number of rows in paths_table has changed
    num_rows = len(paths_table)
    if prev_num_rows is not None and num_rows == prev_num_rows:
        print("Finished! Total number of iterations:", iteration-1) 
        break  # Exit the loop if no change in the number of rows
        
    prev_num_rows = num_rows 
       
    # Extract end points from paths_table (in skan, branch-type==0 is isolated paths, 1 is end paths, 2 is proximal paths) 
    # Here we make sure to select only end paths 
    end_paths = paths_table[paths_table['branch-type']==1]
    print("total end branches:", len(end_paths))
    end_path_ids = end_paths['path-id'].tolist()
       
    coords_list = []
    for path_id in end_path_ids:
        coords_list.extend(skeleton.path_coordinates(int(path_id)))
       
    columns = ['z', 'x', 'y']
    end_point_coords = pd.DataFrame(coords_list, columns=columns)
    for index, row in end_point_coords.iterrows():
        x, y, z = int(row['x']), int(row['y']), int(row['z'])
        endpoint_skel[z, x, y] = 100
       
    # Look for junction pixels, we want to make sure NOT to delete these 
    junction_img = skan.csr.make_degree_image(skeleton_img)
       
    # Process each slice and save the resulting image stack
    z = skeleton_img.shape[0]
    
    with tf.TiffWriter(os.path.join(folder_path,  f"proximal_pruned_{prox_width}.tif")) as tif:
        for im in range(z):
            skeleton_slice = binary_skeleton[im, :, :]
            junction_slice = junction_img[im, :, :]
            endpoint_slice = endpoint_skel[im, :, :]
            skeleton_new = np.copy(skeleton_slice)
            proximal_slice = proximal_img[im, :, :]
            skeleton_new[(proximal_slice == 255) & (endpoint_slice == 100) & (junction_slice <= 2)] = 0
            filename = f"pruned_skeleton_{im}"
            tif.write(skeleton_new, description=filename, metadata=None)

In [ ]:
# If you want to add an additional root generation, you can do this in FIJI with the paintbrush tool
# Next, check that the proximal pruned skeleton is rid of spurious branches 
import napari
viewer = napari.Viewer(ndisplay=3)
prox_pruned = tf.imread(folder_path + f"proximal_pruned_{prox_width}.tif")
viewer.add_image(prox_pruned, name = ('proximal_pruned'))
viewer.add_image(prox_img, name = ('proximal_img'), opacity = 0.4)

In [ ]:
################## OPTIONAL #################################
# optional DELETE paths connected to a node 
# sometimes I can see there are some paths that are obviously artifacts that I was not able to address with previous approaches. 
# I can find the end nodes of such branches, find the corresponding end path and delete it 
# This is helpful for very small samples where it is easy to see what nodes are artifacts. 

import pandas as pd 
import cv2
import tifffile as tf
import numpy as np
import pandas as pd
import re
from skimage.morphology import skeletonize
import skan 
import os

#folder_path = r"F:\Megumi\Dropbox (DBOX-EQS1)\Megumi-Cadisha\Skeleton by Lobe\Left Upper Lobe\proximal_pruning\GW13aLU-P11\B4and5\Trial 2\deloop5\deloop10\proximal_pruned\\"
skeleton_img = prox_pruned
#voxel = int(7)

# we write out nodes of interest
delete_nodes = (90, 91, 87, 88, 92) 
node_positions = pd.read_csv(folder_path + "node_positions.csv", delimiter = ",", names=["Node_ID", "Y", "X", "Z"])

#node_positions.set_axis(["Node_ID", "Y", "X", "Z"], axis=1, inplace=True)
# Sanity check 
print(node_positions.head())

# We will use this function to extract the node positions for a user-specified node 
def find_coordinates(node_positions, node):
    node_data = node_positions.loc[node_positions["Node_ID"] == int(node)]
    x, y, z = node_data.iloc[0][["X", "Y", "Z"]]
    return x, y, z

# Create a DataFrame for node positions
output_df = []

for node in delete_nodes:
    print(node)
    x, y, z = find_coordinates(node_positions, node)
    output_df.append([node, x, y, z])
    
# Creating the DataFrame
columns = ["ID", "X", "Y", "Z"]
output_df = pd.DataFrame(output_df, columns=columns)

# Save the DataFrame to a CSV file with header
output_df.to_csv(folder_path + 'delete_nodes.csv', index=False, header=True)

######### 2. Convert skeleton to paths with skan  #########
#voxel = int(4) 
binary_skeleton = skeletonize(skeleton_img) 
skeleton = skan.Skeleton(binary_skeleton, spacing = voxel)

# Create a paths table for the skeleton 
all_paths = [
    skeleton.path_coordinates(i)
    for i in range(skeleton.n_paths)
    ]
paths_table = skan.summarize(skeleton) 

######### 3. Match node positions to paths  #########
# Initialize an empty list to store path-ids
matched_path_ids = []

# We set the tolerance to 1.5. We can imagine that for a given node, it is involved in 2 paths but in skan, it would only show 1 of the 2.
# Thus, we set the tolerance to a higher number so that we can get both paths involved. 
# From my tests, the optimal value is 1.5. See loops tolerance slides for more. 
tolerance = 1

# Iterate through each row of df1
for index, row in output_df.iterrows():
    # Extract XYZ values from the current row of df1
    XYZ_values_src = [row["X"], row["Y"], row["Z"]]
    XYZ_values_dst = [row["X"], row["Y"], row["Z"]]  # Assuming the same XYZ values for src and dst
    
    # Check if XYZ values match with any row in df2 (src coordinates)
    matching_rows_src = paths_table[((paths_table["image-coord-src-0"] - XYZ_values_src[2]).abs() <= tolerance) &
                             ((paths_table["image-coord-src-1"] - XYZ_values_src[1]).abs() <= tolerance) &
                             ((paths_table["image-coord-src-2"] - XYZ_values_src[0]).abs() <= tolerance)]
    
    # Check if XYZ values match with any row in df2 (dst coordinates)
    matching_rows_dst = paths_table[((paths_table["image-coord-dst-0"] - XYZ_values_dst[2]).abs() <= tolerance) &
                             ((paths_table["image-coord-dst-1"] - XYZ_values_dst[1]).abs() <= tolerance) &
                             ((paths_table["image-coord-dst-2"] - XYZ_values_dst[0]).abs() <= tolerance)]
    
    # If a match is found in either src or dst coordinates, append the corresponding path-id to matched_path_ids
    if not matching_rows_src.empty:
        matched_path_ids.extend(matching_rows_src.index.tolist())
    elif not matching_rows_dst.empty:
        matched_path_ids.extend(matching_rows_dst.index.tolist())

# Print the matched path-ids
print("Matched path-ids:", matched_path_ids)

######### 4. Highlight in skeleton image  #########
delete_skel = np.zeros_like(binary_skeleton)

# Initialize coords_list
coords_list = []
for path_id in matched_path_ids: 
    coords_list.extend(skeleton.path_coordinates(int(path_id)))
    
#print(coords_list) 

columns = ['Z', 'Y', 'X']
delete_coords = pd.DataFrame(coords_list, columns = columns) 
print(delete_coords)

for index, row in delete_coords.iterrows():
    x, y, z = int(row['X']), int(row['Y']), int(row['Z'])
    delete_skel[z, y, x] = 255

# Look for junction pixels, we want to make sure NOT to delete these 
junction_img = skan.csr.make_degree_image(skeleton_img)

# If you want to save the loops_paths as an image stack, uncomment: 
tf.imwrite(folder_path + "delete.ome.tif", delete_skel)

######### 5. Delete loop pixels in skeleton image  #########

# Process each slice and save the resulting image stack
z = skeleton_img.shape[0]
with tf.TiffWriter(os.path.join(folder_path, f"proximal_pruned_{prox_width}.tif")) as tif:
    for im in range(z):
        skeleton_slice = binary_skeleton[im, :, :]
        delete_slice = delete_skel[im, :, :]
        junction_slice = junction_img[im, :, :]
        skeleton_new = np.copy(skeleton_slice)
        skeleton_new[(delete_slice == 255) & (junction_slice <= 2)] = 0
        filename1 = f"cleaned_skeleton_{im}"
        tif.write(skeleton_new, description=filename1, metadata=None)